# Workload Scaling Analysis (Capacity-Aware)

This notebook revisits the workload scaling study with capacity-aware cost and latency metrics.

**Enhancements implemented:**
- Capacity checks using sustainable requests/hour per instance
- Full-hour billing with required instance counts and scaling
- Queue growth risk and utilization flags
- Latency (p50/p95) and failure-rate tracking for SLA validation
- Explicit reporting of when multiple instances are required
- Extended workload sweep up to 50,000 requests/hour


In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
plt.rcParams["figure.figsize"] = (16, 10)
plt.rcParams["font.size"] = 11
sns.set_palette("tab10")

print("Libraries imported.")


In [ ]:
DATA_PATH = Path("parsed_logs/night_logs_6_with_model_info.json")

COST_ASSUMPTIONS = {
    ("cpu", 1): {"hourly_cost": 0.05},
    ("cpu", 2): {"hourly_cost": 0.10},
    ("cpu", 4): {"hourly_cost": 0.20},
    ("cpu", 8): {"hourly_cost": 0.40},
    ("cuda", 25): {"hourly_cost": 0.50},
    ("cuda", 50): {"hourly_cost": 1.00},
    ("cuda", 75): {"hourly_cost": 1.50},
    ("cuda", 100): {"hourly_cost": 2.00},
}

WORKLOAD_LEVELS = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]

DEFAULT_TOKENS_PER_REQUEST = 100
WORKLOAD_TOKEN_PROFILE = {}  # Override per workload if needed

SLA_P95_SECONDS = 2.0  # Target end-to-end response-time SLA

def tokens_for_workload(requests_per_hour: int) -> float:
    return WORKLOAD_TOKEN_PROFILE.get(requests_per_hour, DEFAULT_TOKENS_PER_REQUEST)


In [ ]:
def get_device_key(record: dict):
    variant = record.get("variant")
    if variant == "cpu":
        return ("cpu", record.get("cpu_cores"))
    if variant == "cuda":
        return ("cuda", record.get("gpu_percentage"))
    return None

def safe_extend(container: list, values):
    if not values:
        return
    for value in values:
        if value is None:
            continue
        container.append(value)

def build_hardware_baseline(path: Path, cost_assumptions: dict, model_quant: str = "Q4_K_M") -> pd.DataFrame:
    with path.open() as handle:
        raw_data = json.load(handle)

    aggregates: dict = {}
    for record in raw_data:
        if record.get("model_quant") != model_quant:
            continue
        key = get_device_key(record)
        if key not in cost_assumptions:
            continue

        stats = aggregates.setdefault(
            key,
            {
                "variant": key[0],
                "config_value": key[1],
                "weight": 0.0,
                "throughput_sum": 0.0,
                "failure_sum": 0.0,
                "failure_weight": 0.0,
                "latency_values": [],
                "ttft_values": [],
            },
        )

        throughput = record.get("throughput_mean")
        if throughput is None or pd.isna(throughput):
            continue

        weight = (
            record.get("successful_requests")
            or record.get("total_requests")
            or record.get("throughput_samples")
            or 1.0
        )

        stats["weight"] += weight
        stats["throughput_sum"] += throughput * weight

        failure_rate = record.get("request_failure_rate_mean")
        if failure_rate is not None and not pd.isna(failure_rate):
            stats["failure_sum"] += failure_rate * weight
            stats["failure_weight"] += weight

        safe_extend(stats["latency_values"], record.get("avg_time_per_request_values"))
        safe_extend(stats["ttft_values"], record.get("time_to_first_token_values"))

    baseline_rows = []
    for key, stats in aggregates.items():
        if stats["weight"] <= 0:
            continue

        variant, config_value = key
        hardware_config = f"CPU {config_value} cores" if variant == "cpu" else f"GPU {config_value}%"

        avg_throughput = stats["throughput_sum"] / stats["weight"]
        failure_rate = (
            stats["failure_sum"] / stats["failure_weight"]
            if stats["failure_weight"] > 0
            else math.nan
        )

        def percentile_or_nan(values: list, percentile: float) -> float:
            if not values:
                return math.nan
            return float(np.percentile(values, percentile))

        baseline_rows.append(
            {
                "hardware_config": hardware_config,
                "variant": variant,
                "config_value": config_value,
                "avg_throughput_tokens_per_s": avg_throughput,
                "request_failure_rate": failure_rate,
                "latency_p50_s": percentile_or_nan(stats["latency_values"], 50),
                "latency_p95_s": percentile_or_nan(stats["latency_values"], 95),
                "ttft_p50_s": percentile_or_nan(stats["ttft_values"], 50),
                "hourly_cost": cost_assumptions[key]["hourly_cost"],
            }
        )

    baseline_df = (
        pd.DataFrame(baseline_rows)
        .sort_values("hardware_config")
        .reset_index(drop=True)
    )
    return baseline_df

baseline_df = build_hardware_baseline(DATA_PATH, COST_ASSUMPTIONS)
print(f"Hardware configs captured: {len(baseline_df)}")
baseline_df


### Capacity-aware workload metrics

In [ ]:
def calculate_workload_metrics(
    baseline_df: pd.DataFrame,
    requests_per_hour: int,
    avg_tokens_per_request: float,
    sla_p95_seconds: float,
) -> pd.DataFrame:
    rows = []
    for _, hw in baseline_df.iterrows():
        throughput = hw["avg_throughput_tokens_per_s"]
        if pd.isna(throughput) or throughput <= 0:
            continue

        tokens_per_request = avg_tokens_per_request
        if tokens_per_request <= 0:
            continue

        sustainable_rph = throughput * 3600.0 / tokens_per_request
        if sustainable_rph <= 0:
            continue

        required_instances = max(1, math.ceil(requests_per_hour / sustainable_rph))
        capacity_rph = sustainable_rph * required_instances
        total_hourly_cost = required_instances * hw["hourly_cost"]

        single_instance_utilization = (
            requests_per_hour / sustainable_rph * 100.0 if sustainable_rph else math.nan
        )
        utilization = requests_per_hour / capacity_rph if capacity_rph > 0 else math.nan
        headroom = max(0.0, 1.0 - utilization) if not math.isnan(utilization) else math.nan

        queue_growth_single = (
            max(0.0, requests_per_hour - sustainable_rph) / requests_per_hour
            if requests_per_hour > 0
            else math.nan
        )

        cost_per_request = (
            total_hourly_cost / requests_per_hour if requests_per_hour > 0 else math.nan
        )

        latency_p95 = hw["latency_p95_s"]
        meets_latency = None
        if isinstance(latency_p95, (int, float)) and not math.isnan(latency_p95):
            meets_latency = latency_p95 <= sla_p95_seconds

        rows.append(
            {
                "requests_per_hour": requests_per_hour,
                "tokens_per_request": tokens_per_request,
                "hardware_config": hw["hardware_config"],
                "variant": hw["variant"],
                "required_instances": required_instances,
                "sustainable_rph_per_instance": sustainable_rph,
                "capacity_rph": capacity_rph,
                "meets_capacity_with_scaling": requests_per_hour <= capacity_rph,
                "meets_single_instance_capacity": requests_per_hour <= sustainable_rph,
                "single_instance_utilization_percent": single_instance_utilization,
                "fleet_utilization_percent": (
                    utilization * 100.0 if not math.isnan(utilization) else math.nan
                ),
                "capacity_headroom_percent": (
                    headroom * 100.0 if not math.isnan(headroom) else math.nan
                ),
                "queue_growth_rate_single_instance_percent": (
                    queue_growth_single * 100.0
                    if not math.isnan(queue_growth_single)
                    else math.nan
                ),
                "total_hourly_cost": total_hourly_cost,
                "cost_per_request": cost_per_request,
                "latency_p50_s": hw["latency_p50_s"],
                "latency_p95_s": latency_p95,
                "ttft_p50_s": hw["ttft_p50_s"],
                "request_failure_rate": hw["request_failure_rate"],
                "meets_latency_sla": meets_latency,
            }
        )

    return pd.DataFrame(rows)


In [ ]:
workload_frames = []
for workload in WORKLOAD_LEVELS:
    tokens = tokens_for_workload(workload)
    metrics = calculate_workload_metrics(
        baseline_df,
        requests_per_hour=workload,
        avg_tokens_per_request=tokens,
        sla_p95_seconds=SLA_P95_SECONDS,
    )
    workload_frames.append(metrics)

combined_df = pd.concat(workload_frames, ignore_index=True)
combined_df.head()


In [ ]:
scaling_required = combined_df[combined_df["required_instances"] > 1].copy()
scaling_required = scaling_required[
    [
        "requests_per_hour",
        "hardware_config",
        "required_instances",
        "capacity_rph",
        "fleet_utilization_percent",
        "cost_per_request",
    ]
]
scaling_required = scaling_required.sort_values(["requests_per_hour", "required_instances", "hardware_config"])
print(f"Configurations requiring scaling: {len(scaling_required)}")
scaling_required.head(10)


In [ ]:
feasible_df = combined_df[combined_df["meets_capacity_with_scaling"]].copy()

plt.figure()
for config in feasible_df["hardware_config"].unique():
    subset = feasible_df[feasible_df["hardware_config"] == config]
    subset = subset.sort_values("requests_per_hour")
    plt.plot(
        subset["requests_per_hour"],
        subset["cost_per_request"],
        marker="o",
        linewidth=3,
        markersize=8,
        label=config,
    )
plt.xscale("log")
plt.yscale("log")
plt.xlabel("Requests per Hour")
plt.ylabel("Cost per Request ($)")
plt.title("Cost per Request vs Workload (Scaled as Needed)")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
for config in feasible_df["hardware_config"].unique():
    subset = feasible_df[feasible_df["hardware_config"] == config]
    subset = subset.sort_values("requests_per_hour")
    plt.step(
        subset["requests_per_hour"],
        subset["required_instances"],
        where="post",
        linewidth=2,
        label=config,
    )
plt.xscale("log")
plt.xlabel("Requests per Hour")
plt.ylabel("Required Instances")
plt.title("Instance Count Needed per Hardware Configuration")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", ncol=1)
plt.tight_layout()
plt.show()


In [ ]:
heatmap_source = feasible_df.copy()
heatmap_data = heatmap_source.pivot_table(
    values="cost_per_request",
    index="hardware_config",
    columns="requests_per_hour",
    aggfunc="min",
)
plt.figure(figsize=(14, 8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".4f",
    cmap="RdYlGn_r",
    cbar_kws={"label": "Cost per Request ($)"},
)
plt.title("Cost per Request Heatmap (Scaled Deployments)")
plt.xlabel("Requests per Hour")
plt.ylabel("Hardware Configuration")
plt.tight_layout()
plt.show()


In [ ]:
def select_optimal_configs(
    metrics_df: pd.DataFrame,
    require_capacity: bool = True,
    require_latency: bool = True,
) -> pd.DataFrame:
    selections = []
    for workload in WORKLOAD_LEVELS:
        subset = metrics_df[metrics_df["requests_per_hour"] == workload]
        if require_capacity:
            subset = subset[subset["meets_capacity_with_scaling"]]
        if require_latency:
            subset = subset[subset["meets_latency_sla"] != False]
        if subset.empty:
            print(f"{workload:5d} req/hr -> no configuration passes the selection filters")
            continue
        best = subset.loc[subset["cost_per_request"].idxmin()]
        latency = best["latency_p95_s"]
        latency_text = (
            f"{latency:.2f}s"
            if isinstance(latency, (int, float)) and not math.isnan(latency)
            else "n/a"
        )
        utilization = best["fleet_utilization_percent"]
        utilization_text = (
            f"{utilization:.1f}%"
            if isinstance(utilization, (int, float)) and not math.isnan(utilization)
            else "n/a"
        )
        print(
            f"{workload:5d} req/hr -> {best['hardware_config']} | "
            f"{best['required_instances']} instance(s) | ${best['cost_per_request']:.4f}/req | "
            f"fleet util {utilization_text} | P95 latency {latency_text}"
        )
        selections.append(
            {
                "requests_per_hour": workload,
                "hardware_config": best["hardware_config"],
                "required_instances": best["required_instances"],
                "cost_per_request": best["cost_per_request"],
                "fleet_utilization_percent": best["fleet_utilization_percent"],
                "latency_p95_s": best["latency_p95_s"],
                "meets_latency_sla": best["meets_latency_sla"],
            }
        )
    if not selections:
        return pd.DataFrame(
            columns=[
                "requests_per_hour",
                "hardware_config",
                "required_instances",
                "cost_per_request",
                "fleet_utilization_percent",
                "latency_p95_s",
                "meets_latency_sla",
            ]
        )
    return pd.DataFrame(selections)

optimal_df = select_optimal_configs(combined_df)
optimal_df


In [ ]:
def compute_savings(metrics_df: pd.DataFrame, optimal_df: pd.DataFrame) -> pd.DataFrame:
    if optimal_df.empty:
        return pd.DataFrame(
            columns=[
                "requests_per_hour",
                "optimal_cost",
                "best_cpu_cost",
                "best_gpu_cost",
                "savings_vs_cpu_percent",
                "savings_vs_gpu_percent",
            ]
        )
    results = []
    for workload in WORKLOAD_LEVELS:
        optimal_row = optimal_df[optimal_df["requests_per_hour"] == workload]
        if optimal_row.empty:
            continue
        optimal_cost = float(optimal_row["cost_per_request"].iloc[0])

        cpu_options = metrics_df[
            (metrics_df["requests_per_hour"] == workload)
            & (metrics_df["variant"] == "cpu")
            & (metrics_df["meets_capacity_with_scaling"])
        ]
        gpu_options = metrics_df[
            (metrics_df["requests_per_hour"] == workload)
            & (metrics_df["variant"] == "cuda")
            & (metrics_df["meets_capacity_with_scaling"])
        ]

        cpu_cost = cpu_options["cost_per_request"].min() if not cpu_options.empty else math.nan
        gpu_cost = gpu_options["cost_per_request"].min() if not gpu_options.empty else math.nan

        cpu_savings = (
            (cpu_cost - optimal_cost) / cpu_cost * 100.0
            if cpu_cost and not math.isnan(cpu_cost) and cpu_cost > 0
            else math.nan
        )
        gpu_savings = (
            (gpu_cost - optimal_cost) / gpu_cost * 100.0
            if gpu_cost and not math.isnan(gpu_cost) and gpu_cost > 0
            else math.nan
        )

        results.append(
            {
                "requests_per_hour": workload,
                "optimal_cost": optimal_cost,
                "best_cpu_cost": cpu_cost,
                "best_gpu_cost": gpu_cost,
                "savings_vs_cpu_percent": cpu_savings,
                "savings_vs_gpu_percent": gpu_savings,
            }
        )
    return pd.DataFrame(results)

savings_df = compute_savings(combined_df, optimal_df)
savings_df


In [ ]:
avg_cpu_savings = savings_df["savings_vs_cpu_percent"].mean()
avg_gpu_savings = savings_df["savings_vs_gpu_percent"].mean()
print("Average savings vs best CPU: " + (f"{avg_cpu_savings:.1f}%" if not math.isnan(avg_cpu_savings) else "n/a"))
print("Average savings vs best GPU: " + (f"{avg_gpu_savings:.1f}%" if not math.isnan(avg_gpu_savings) else "n/a"))


In [ ]:
summary_rows = []
for workload in WORKLOAD_LEVELS:
    subset = combined_df[combined_df["requests_per_hour"] == workload]
    requires_scaling = (subset["required_instances"] > 1).sum()
    latency_failures = (subset["meets_latency_sla"] == False).sum()
    summary_rows.append(
        {
            "requests_per_hour": workload,
            "configs_requiring_scaling": int(requires_scaling),
            "latency_sla_failures": int(latency_failures),
        }
    )
pd.DataFrame(summary_rows)


In [ ]:
combined_df.to_csv("workload_scaling_capacity_metrics.csv", index=False)
optimal_df.to_csv("optimal_hardware_capacity_selection.csv", index=False)
savings_df.to_csv("workload_scaling_capacity_savings.csv", index=False)
print("Results exported to CSV.")
